In [9]:
import sys
import os

sys.path.insert(0, os.path.abspath("../src"))

In [10]:
import pandas as pd

from profile_data import (
    engine,
    get_tables,
    load_data,
    profile_table
)

print("profile_data imported successfully!")

profile_data imported successfully!


In [11]:
tables_df = get_tables(engine)

print(f"Total tables: {len(tables_df)}")

tables_df

Total tables: 19


,table_name
0,other_american_b01362
1,other_carmel_b00256
2,other_dial7_b00887
3,other_diplo_b01196
4,other_federal_02216
5,other_fhv_services_jan_aug_2015
6,other_firstclass_b01536
7,other_highclass_b01717
8,other_lyft_b02510
9,other_prestige_b01338


In [12]:
table_summary = []

for table_name in tables_df["table_name"]:

    df = load_data(engine, table_name)

    table_summary.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

table_summary = pd.DataFrame(table_summary)

table_summary

,table_name,rows,columns
0,other_american_b01362,91712,6
1,other_carmel_b00256,256519,4
2,other_dial7_b00887,194992,6
3,other_diplo_b01196,98550,3
4,other_federal_02216,276,7
5,other_fhv_services_jan_aug_2015,26181,5
6,other_firstclass_b01536,166769,3
7,other_highclass_b01717,151925,3
8,other_lyft_b02510,267701,4
9,other_prestige_b01338,320641,3


In [13]:
table_summary.sort_values(
    by="rows",
    ascending=False
)

,table_name,rows,columns
14,uber_raw_data_janjune_15,14270479,4
18,uber_raw_data_sep14,1028136,4
13,uber_raw_data_aug14,829275,4
15,uber_raw_data_jul14,796121,4
16,uber_raw_data_jun14,663844,4
17,uber_raw_data_may14,652435,4
12,uber_raw_data_apr14,564516,4
9,other_prestige_b01338,320641,3
8,other_lyft_b02510,267701,4
1,other_carmel_b00256,256519,4


In [15]:
schema_query = """
SELECT
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
"""

schema_df = pd.read_sql(schema_query, engine)

schema_df

,table_name,ordinal_position,column_name,data_type,is_nullable
0,other_american_b01362,1,DATE,text,YES
1,other_american_b01362,2,TIME,text,YES
2,other_american_b01362,3,PICK UP ADDRESS,text,YES
3,other_american_b01362,4,Unnamed: 3,double precision,YES
4,other_american_b01362,5,Unnamed: 4,double precision,YES
...,...,...,...,...,...
77,uber_raw_data_may14,4,Base,text,YES
78,uber_raw_data_sep14,1,Date/Time,text,YES
79,uber_raw_data_sep14,2,Lat,double precision,YES
80,uber_raw_data_sep14,3,Lon,double precision,YES


In [16]:
print(f"Total tables  : {schema_df['table_name'].nunique()}")
print(f"Total columns : {len(schema_df)}")

Total tables  : 19
Total columns : 82


In [17]:
for table_name, group in schema_df.groupby("table_name"):

    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)

    print(
        group[
            [
                "column_name",
                "data_type",
                "is_nullable"
            ]
        ].to_string(index=False)
    )


other_american_b01362
    column_name        data_type is_nullable
           DATE             text         YES
           TIME             text         YES
PICK UP ADDRESS             text         YES
     Unnamed: 3 double precision         YES
     Unnamed: 4 double precision         YES
     Unnamed: 5 double precision         YES

other_carmel_b00256
column_name data_type is_nullable
       Date      text         YES
       Time      text         YES
  PU_Adress      text         YES
    Base_No      text         YES

other_dial7_b00887
column_name data_type is_nullable
       Date      text         YES
       Time      text         YES
      State      text         YES
     PuFrom      text         YES
    Address      text         YES
     Street      text         YES

other_diplo_b01196
column_name data_type is_nullable
       Date      text         YES
       Time      text         YES
 PU_Address      text         YES

other_federal_02216
    column_name data_type is_nullabl

In [18]:
def show_sample_data(engine, table_name, n=5):

    query = f'''
    SELECT *
    FROM "{table_name}"
    LIMIT {n};
    '''

    return pd.read_sql(query, engine)

In [19]:
show_sample_data(
    engine,
    "other_american_b01362",
    10
)

,DATE,TIME,PICK UP ADDRESS,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,7/1/2014,12:00:00 AM,"874 E 139th St Mott Haven, BX",None,None,None
1,7/1/2014,12:01:00 AM,"628 E 141st St Mott Haven, BX",None,None,None
2,7/1/2014,12:01:00 AM,"601 E 156th St South Bronx, BX",None,None,None
3,7/1/2014,12:01:00 AM,"708 E 138th St Mott Haven, BX",None,None,None
4,7/1/2014,12:02:00 AM,"700 E 140th St Mott Haven, BX",None,None,None
5,7/1/2014,12:03:00 AM,"514 E 163rd St Cortlandt, BX",None,None,None
6,7/1/2014,12:08:00 AM,"300 E 150th St Cortlandt, BX",None,None,None
7,7/1/2014,12:10:00 AM,"370 E 153rd St South Bronx, BX",None,None,None
8,7/1/2014,12:11:00 AM,"455 E 148th St South Bronx, BX",None,None,None
9,7/1/2014,12:11:00 AM,"600 E 141st St Mott Haven, BX",None,None,None


In [20]:
show_sample_data(
    engine,
    "other_carmel_b00256",
    10
)

,Date,Time,PU_Adress,Base_No
0,7/1/2014,0:00,260 W 44 St NYC,B00256
1,7/1/2014,0:00,125 W 29 St Nyc,B00256
2,7/1/2014,0:00,141 W 28 St Nyc,B00256
3,7/1/2014,0:01,EWR,B00256
4,7/1/2014,0:07,JFK,B00256
5,7/1/2014,0:07,EWR,B00256
6,7/1/2014,0:11,JFK,B00256
7,7/1/2014,0:12,EWR,B00256
8,7/1/2014,0:15,EWR,B00256
9,7/1/2014,0:15,206 W 109 St Nyc,B00256


In [21]:
show_sample_data(
    engine,
    "other_dial7_b00887",
    10
)

,Date,Time,State,PuFrom,Address,Street
0,2014.07.06,14:30,NY ...,MANHATTAN,50,MURRAY ST
1,2014.07.04,7:15,NY ...,MANHATTAN,143,AVENUE B
2,2014.07.05,5:45,NY ...,MANHATTAN,125,CHRISTOPHER ST
3,2014.07.06,4:30,NY ...,MANHATTAN,217,E 7 ST
4,2014.07.05,7:45,NY ...,MANHATTAN,521,W 26 ST
5,2014.07.06,8:45,NY ...,MANHATTAN,110,BLEECKER ST
6,2014.07.06,6:45,NY ...,MANHATTAN,220,E 57 St
7,2014.07.06,11:30,NY ...,MANHATTAN,440,E 20 ST
8,2014.07.03,12:00,NY ...,QUEENS,2204,119 ST
9,2014.07.03,13:00,NY ...,MANHATTAN,286,E 10 ST
